# Lab 01 — Sistemas de Controle e Modelos Matemáticos

**Módulo 01 · Semana 2 · Laboratório de Controle Automático (Ifes Guarapari)**

Neste laboratório você vai:
1. Simular a resposta de sistemas de 1ª e 2ª ordem ao degrau;
2. Construir modelos massa-mola, RLC e motor CC a partir das equações físicas;
3. Verificar o efeito dos polos na forma da resposta.

**Teoria de apoio:** `teoria_modulo1.md`, §1.1 (modelos) e §1.2 (função de transferência).

> Convenção do curso: sistemas SISO, entrada u, saída y, `sen` = seno, `s` = variável de Laplace.

In [ ]:
# %% IMPORTS (rode esta célula primeiro)
!pip install --quiet control==0.10.2
import control as ct
import numpy as np
import matplotlib.pyplot as plt
s = ct.tf('s')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
print('python-control', ct.__version__)

## Parte 1 — Primeiro contato: simulando uma função de transferência

No `python-control`, criamos uma FT com `ct.tf(numerador, denominador)` e simulamos
a resposta ao degrau com `ct.step_response`. Vamos começar pelo sistema de 1ª ordem
G(s) = 1/(s + 1) e comparar com G(s) = 1/(s + 5). Pergunta-guia: **qual acomoda mais rápido e por quê?**

In [ ]:
G1 = ct.tf(1, [1, 1])   # 1/(s+1)
G5 = ct.tf(1, [1, 5])   # 1/(s+5)
t = np.linspace(0, 6, 600)
t1, y1 = ct.step_response(G1, t)
t5, y5 = ct.step_response(G5, t)
plt.plot(t1, y1, label='G(s) = 1/(s+1)')
plt.plot(t5, y5, label='G(s) = 1/(s+5)')
plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.legend()
plt.title('Resposta ao degrau — 1ª ordem'); plt.show()
print('polos:', ct.poles(G1), ct.poles(G5))

## Parte 2 — Construindo os três modelos do curso

Abaixo estão os parâmetros físicos dos três sistemas-modelo (§1.1.4). A partir das EDOs
deduzidas em sala, monte as FTs copiando os coeficientes:

- massa-mola: ÿ + (B/M)ẏ + (K/M)y = (1/M)u
- RLC: v̈C + (R/L)v̇C + (1/LC)vC = (1/LC)v
- motor CC (velocidade): ω̈ + (R/L)ω̇ + (k²/LJ)ω = (k/LJ)u

In [ ]:
# Massa-mola-amortecedor
M, B, K = 1.0, 5.0, 6.0
Gmm = ct.tf([1/M], [1, B/M, K/M])
print('massa-mola:', Gmm)

# RLC série (saída: tensão no capacitor)
R, L, C = 2.0, 1.0, 0.5
Grlc = ct.tf([1/(L*C)], [1, R/L, 1/(L*C)])
print('RLC:', Grlc)

# Motor CC (saída: velocidade angular)
R_, L_, k_, J_ = 2.0, 0.5, 1.0, 0.25
Gmot = ct.tf([k_/(L_*J_)], [1, R_/L_, k_**2/(L_*J_)])
print('motor CC:', Gmot)

t = np.linspace(0, 8, 800)
for G, nome in [(Gmm,'massa-mola'), (Grlc,'RLC'), (Gmot,'motor CC')]:
    tt, yy = ct.step_response(G, t)
    plt.plot(tt, yy, label=f'{nome}  polos={np.round(ct.poles(G),2)}')
plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.legend()
plt.title('Os três modelos — resposta ao degrau unitário'); plt.show()

## Parte 3 — Sua vez

**Exercício L1.1.** Varie B no massa-mola (tente B = 1 e B = 7) e descreva o que acontece
com a forma da resposta (oscila? ultrapassa?). Relacione com a posição dos polos.

**Exercício L1.2.** O motor CC com saída em **posição** (em vez de velocidade) acrescenta
um integrador: Gp(s) = Gmot(s)/s. Simule a resposta ao degrau de Gp em malha aberta.
O que acontece com y(t)? Esse sistema é BIBO-estável? (§1.3.1)

**Exercício L1.3.** Desafio do controle remoto: encontre (tentativa e erro, 3 casas) o valor
de B que torna o massa-mola **criticamente amortecido** (polos reais iguais, Δ = 0).